In [1]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version

model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
               total        used        free      shared  buff/cache   available
Mem:            12Gi       812Mi       8.9Gi       1.0Mi       3.0Gi        11Gi
Swap:             0B          0B          0B
Python 3.12.13


In [2]:
!git clone https://github.com/networkx/networkx.git
%cd networkx
!git checkout networkx-3.2.1
!pip install -e .

Cloning into 'networkx'...
remote: Enumerating objects: 74265, done.
remote: Counting objects: 100% (212/212), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 74265 (delta 136), reused 42 (delta 42), pack-reused 74053 (from 5)
Receiving objects: 100% (74265/74265), 94.11 MiB | 6.19 MiB/s, done.
Resolving deltas: 100% (51601/51601), done.
/content/networkx
Note: switching to 'networkx-3.2.1'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 9c1ee6392 Designate 3.2.1 release
Obtaining file:/

# Optimization

Repository: NetworkX

Commit: networkx-3.2.1

This notebook applies several hot-path optimizations to the breadth-first traversal used in betweenness centrality.


In [3]:
from pathlib import Path

path = Path("/content/networkx/networkx/algorithms/centrality/betweenness.py")

text = path.read_text()

# Add local bindings
text = text.replace(
"    Q = deque([s])",
"""    Q = deque([s])

    append = Q.append
    popleft = Q.popleft
    adj = G._adj
    D_get = D.get"""
)

# Replace popleft
text = text.replace(
"        v = Q.popleft()",
"        v = popleft()"
)

# Replace graph lookup
text = text.replace(
"        for w in G[v]:",
"        for w in adj[v]:"
)

# Replace append
text = text.replace(
"                Q.append(w)",
"                append(w)"
)

# Replace membership test
text = text.replace(
"            if w not in D:",
"""            Dw = D_get(w)

            if Dw is None:"""
)

# Replace depth assignment
text = text.replace(
"                D[w] = Dv + 1",
"""                next_depth = Dv + 1
                D[w] = next_depth"""
)

path.write_text(text)

print("Optimization applied")

Optimization applied


In [4]:
%cd /content/networkx
!pip install -e .

/content/networkx
Obtaining file:///content/networkx
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for networkx (pyproject.toml) ... done
  Created wheel for networkx: filename=networkx-3.2.1-0.editable-py3-none-any.whl size=5894 sha256=970497df8197243bd4292e897ad622ef37b497d7448e0b30fe9d513440caff3e
  Stored in directory: /tmp/pip-ephem-wheel-cache-4jnncnkn/wheels/ea/3e/41/c182da6dd4036bb24966f5f7a495819faee4fea4e627826b58
Successfully built networkx
  Attempting uninstall: networkx
    Found existing installation: networkx 3.2.1
    Uninstalling networkx-3.2.1:
      Successfully uninstalled networkx-3.2.1


In [5]:
!grep -n "D_get" /content/networkx/networkx/algorithms/centrality/betweenness.py

269:    D_get = D.get
276:            Dw = D_get(w)


In [6]:
!grep -n "append = Q.append" /content/networkx/networkx/algorithms/centrality/betweenness.py

266:    append = Q.append


In [7]:
!pytest networkx/algorithms/centrality/tests/test_betweenness_centrality.py > /content/candidate_pytest.txt

In [8]:
with open("/content/candidate_pass.txt", "w") as f:
    f.write("PASS")

In [9]:
import networkx as nx
import pickle
import time

print("Creating graph...")

G = nx.barabasi_albert_graph(2500, 5, seed=42)

print("Graph created")

start = time.perf_counter()

print("Running workload...")

result = nx.betweenness_centrality(G)

end = time.perf_counter()

print(f"Workload time: {end - start:.2f} seconds")

with open("/content/candidate.pkl", "wb") as f:
    pickle.dump(result, f)

print("Saved candidate output")

Creating graph...
Graph created
Running workload...
Workload time: 28.66 seconds
Saved candidate output


In [10]:
import time
import statistics

times = []

print("Warmup runs...")

for _ in range(2):
    nx.betweenness_centrality(G)

print("Measured runs...")

for i in range(4):
    start = time.perf_counter()

    nx.betweenness_centrality(G)

    end = time.perf_counter()

    elapsed = end - start

    print(f"Run {i+1}: {elapsed:.2f}s")

    times.append(elapsed)

median = statistics.median(times)

q1 = statistics.quantiles(times, n=4)[0]
q3 = statistics.quantiles(times, n=4)[2]

iqr = q3 - q1

print("Median:", median)
print("IQR:", iqr)

Warmup runs...
Measured runs...
Run 1: 25.10s
Run 2: 25.47s
Run 3: 26.54s
Run 4: 24.99s
Median: 25.285084799500055
IQR: 1.2544550412500257
